# 🛵 GUARDIAN — 베트남 야시장 정지 바이크 필터링 (v8b.3 최종본)

**프로젝트:** AIFFEL ML 과정 — 모터사이클 ADAS HUD
**작성자:** 종현
**환경:** Google Colab Pro / A100 GPU (또는 T4 가능)
**입력:** `2345 (2).mp4` — 베트남 야시장 28초 영상

---

## 📋 과제

인도/도로 가장자리에 줄지어 정지된 바이크들이 객체 탐지에서 다 잡히는데, HUD 시스템에서 이들을 무시하고 **주행 중인 객체만** 인식해야 함.

| 분류 | 처리 |
|---|---|
| 차도 위 움직이는 바이크 | ✅ 인식 |
| 차도/인도 위 움직이는 사람 | ✅ 인식 |
| 차 + 차 안 사람 | ✅ 인식 |
| **인도/도로 가장자리 정지 바이크** | ❌ 무시 |

---

## 🤖 사용한 모델 4개

| # | 모델 | 출처 | 역할 |
|---|---|---|---|
| 1 | **YOLO11m-seg** | Ultralytics | 객체 탐지 |
| 2 | **SegFormer-B5** | `nvidia/segformer-b5-finetuned-cityscapes-1024-1024` | 인도(sidewalk) 의미 분할 |
| 3 | **YOLOPv2** | `CAIC-AD/YOLOPv2` | 도로(drivable area) + 차선 |
| 4 | **Farneback Optical Flow** | OpenCV | 모션 (v8b만 사용, 후속 폐기) |

> 모델 추론은 **모두 사전에 1회만 수행** → 캐시. 본 노트북은 캐시만 재활용.

---

## 📈 알고리즘 진화 — 3단계

| 버전 | 사용 신호 | 필터된 정지 바이크 | 평가 |
|---|---|---|---|
| **v8b** | Optical Flow motion만 | **1,159건** (4%) | ❌ 실패 (parallax 문제) |
| **v8b.2** | + Sidewalk(B5) + 위치 + 라이더 체크 | **14,769건** (50%) | ⚠️ 절반만 |
| **v8b.3** ⭐ | + DA mask(YOLOPv2) + Temporal smoothing | **19,487건** (66%) | ✅ 발표/시제품 적합 |

총 motorcycle 객체 29,327건 / 1,655 프레임 / 28초 영상

---

## 🔍 핵심 발견 — Motion만으로는 못 푸는 이유

진단 셀에서 측정한 motion_score 분포:

```
좌/우 가장자리 (정지 의심): 중앙값 89.4
중앙 (주행 중)            : 중앙값 19.6
```

**정지된 바이크가 주행 중 바이크보다 모션이 4.5배 높음** ← 카메라가 진행하면서 가장자리 정지 객체가 시야에서 빠르게 흘러감 (parallax 효과). residual flow 보정으로도 못 메꿈.

→ 이 진단으로 **motion 단독 필터를 버리고 sidewalk + 위치 + DA mask 조합으로 전환**.


## 1️⃣ 환경 설정 + 캐시 경로

모든 모델 추론은 사전 완료. 여기서는 캐시만 재사용.

In [ ]:
# ============================================================
# 환경 설정 + 캐시 경로
# ============================================================
import cv2, json, numpy as np, math, time, os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# Google Drive 마운트 (Colab인 경우)
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

# 디렉토리 (모든 캐시는 Drive)
VIDEO_DIR = Path('/content/drive/MyDrive/video/2')
CACHE     = Path('/content/drive/MyDrive/GUARDIAN/cache')
OUT_DIR   = Path('/content/drive/MyDrive/GUARDIAN/output')
OUT_DIR.mkdir(exist_ok=True, parents=True)

# 입력 파일 (베트남 야시장)
VIDEO    = VIDEO_DIR / '2345 (2).mp4'                    # 원본 영상 (3840×2160, 28초)
YOLO_J   = CACHE / 'yolo_json'  / '2345_2_yolo.json'     # YOLO11m-seg 결과
SEG_B5   = CACHE / 'seg_npz_b5' / '2345_2_seg_b5.npz'    # SegFormer-B5 결과
YOLOP    = CACHE / 'yolop_npz'  / '2345_2_yolop.npz'     # YOLOPv2 결과
FLOW     = CACHE / 'flow_npz'   / '2345_2_flow.npz'      # Farneback Flow (미사용)
OUT_PATH = OUT_DIR / 'v8b3_vietnam_yashijang.mp4'        # 최종 출력
print('✅ 경로 설정 완료')

## 2️⃣ 캐시 로드

In [ ]:
# ============================================================
# 캐시 로드 (4개 모델의 사전 추론 결과)
# ============================================================
print("📂 캐시 로드 중...")

# [모델 1] YOLO11m-seg 객체 탐지 결과
#   Ultralytics YOLO11m-seg, conf=0.15, imgsz=960
#   클래스: motorcycle, person, car, truck, bus 등 COCO 80개
with open(YOLO_J) as f:
    yolo = json.load(f)
orig_w, orig_h = yolo['width'], yolo['height']
fps = yolo['fps']
n_frames_yolo = len(yolo['frames'])
print(f"  [1] YOLO         : {n_frames_yolo} 프레임, {orig_w}×{orig_h} @ {fps}fps")

# [모델 2] SegFormer-B5 sidewalk 마스크
#   nvidia/segformer-b5-finetuned-cityscapes-1024-1024
#   Cityscapes 19 클래스 중 sidewalk(=1)만 추출 → np.packbits 압축 저장
seg_raw = np.load(SEG_B5)
sn, sh, sw = seg_raw['shape']
side_masks = np.unpackbits(seg_raw['sidewalk'])[:sn*sh*sw].reshape(sn, sh, sw)
print(f"  [2] SegFormer-B5 : {sn} 프레임, {sw}×{sh} (sidewalk)")

# [모델 3] YOLOPv2 (CAIC-AD)
#   도로 영역(drivable area, da_masks) + 차선(ll_masks)
#   본 노트북에서는 da_masks만 사용
yolop_raw = np.load(YOLOP)
yn, yh, yw = yolop_raw['shape']
da_masks_raw = np.unpackbits(yolop_raw['da_masks'])[:yn*yh*yw].reshape(yn, yh, yw)
print(f"  [3] YOLOPv2      : {yn} 프레임, {yw}×{yh} (drivable area)")

# [모델 4] Farneback Optical Flow — v8b.3에서는 미사용
#   v8b 첫 시도에서 사용했으나 parallax 문제로 폐기
print(f"  [4] OpticalFlow  : 캐시 있음 (v8b.3에서는 미사용 — parallax 문제)")

## 3️⃣ 마스크 전처리 — Sidewalk Dilate + DA Erode

야간 + 노점 가림 + 사람 군집으로 SegFormer-B5도 인도 영역을 작게 잡음 (전체의 3.7%만).

- **Sidewalk dilate (50px)** — 인도 영역 확장 → 인도 판정 더 관대하게
- **Drivable area erode (7×7)** — 도로 좁힘 → 가장자리 정지 바이크가 "비도로"로 잡히게

In [ ]:
# ============================================================
# 마스크 전처리: Sidewalk Dilate + DA Erode
# ============================================================
# Sidewalk dilate: 50px 등가 (마스크 크기에 비례)
print("⚙️  Sidewalk dilate (50px equivalent)...")
kernel_size = max(7, int(50 * sw / orig_w))
if kernel_size % 2 == 0:
    kernel_size += 1
kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
side_masks_dilated = np.zeros_like(side_masks)
for i in range(sn):
    side_masks_dilated[i] = cv2.dilate(side_masks[i], kernel_dilate, iterations=1)
print(f"   인도 픽셀 비율: {side_masks.mean()*100:.1f}% → {side_masks_dilated.mean()*100:.1f}%")

# YOLOPv2 da_masks erode: 도로 영역 좁힘
print("⚙️  Drivable area erode (7×7)...")
kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
da_eroded = np.zeros_like(da_masks_raw)
for i in range(yn):
    da_eroded[i] = cv2.erode(da_masks_raw[i], kernel_erode, iterations=1)
print(f"   도로 비율: {da_masks_raw.mean()*100:.1f}% → {da_eroded.mean()*100:.1f}%")

## 4️⃣ 핵심 알고리즘 — `is_parked_v4` (motion-free)

### 진화 비교

| 버전 | 사용 신호 | 핵심 변경 |
|---|---|---|
| **v8b** (실패) | motion (Farneback flow_thr) | motion 단독 — parallax로 실패 |
| **v8b.2** | + sidewalk(B5) + 위치 + 라이더 | motion 버림, 위치 fallback 추가 |
| **v8b.3** ⭐ | + **DA(YOLOPv2)** + **Temporal ±2프레임** | 도로 비율 < 40% → 주차, 시간 평균 안정화 |

### 판정 순서

1. **라이더 4중 체크** → 라이더 있으면 주행 중 (False)
2. **DA(도로) 비율 < 40%** → 비도로 = 주차 (True)  *NEW v8b.3*
3. **Sidewalk 비율 > 10%** → 인도 위 = 주차 (True)
4. **화면 가장자리 좌/우 20%** → 주차 (sidewalk fallback)

In [ ]:
# ============================================================
# 헬퍼: 박스 영역의 마스크 비율 (temporal smoothing)
# ============================================================
def get_pct_smooth(masks, fi, x1, y1, x2, y2, window=2):
    """
    박스 영역에서 ±window 프레임 평균 비율 계산.
    v8b.3에서 추가: 단일 프레임 노이즈/깜빡임 안정화.
    """
    n = len(masks)
    lo, hi = max(0, fi - window), min(n, fi + window + 1)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    pcts = []
    for i in range(lo, hi):
        pcts.append(float(masks[i][y1:y2, x1:x2].mean()))
    return float(np.mean(pcts)) if pcts else 0.0


# ============================================================
# is_parked_v4 — 최종 알고리즘 (v8b.3, motion-free)
# ============================================================
def is_parked_v4(box, all_objects, side_masks_d, da_masks_e, fi,
                 orig_w, orig_h,
                 side_thr=0.10, edge_zone=0.20, road_thr=0.40,
                 sn_lim=None, yn_lim=None):
    """
    바이크가 주차되어 있는지 판정 (motion-free).
    
    Args:
        box          : [x1, y1, x2, y2] motorcycle 박스
        all_objects  : 같은 프레임의 모든 객체 (라이더 체크용)
        side_masks_d : dilate된 sidewalk 마스크 (n_frames, h, w)
        da_masks_e   : erode된 drivable area 마스크 (n_frames, h, w)
        fi           : 프레임 인덱스
        side_thr     : sidewalk 비율 임계 (이상 → 주차)
        edge_zone    : 가장자리 비율 (좌/우 20%)
        road_thr     : 도로 비율 임계 (이하 → 주차)
    
    Returns:
        bool — True면 주차, False면 주행 중
    """
    x1, y1, x2, y2 = box
    mh_box = y2 - y1
    mw_box = x2 - x1

    # 너무 작은 박스 (먼 객체) — 보수적 (주차 판정 X)
    if mh_box * mw_box < (orig_w * orig_h) * 0.0008:
        return False

    # ──────────────────────────────────────────────
    # [1] 라이더 4중 체크 (v8b.2 추가)
    # ──────────────────────────────────────────────
    # 사람이 바이크 위에 "앉은" 패턴만 라이더로 인정
    # → 옆에 서있는 보행자가 라이더로 오인되는 것 방지
    has_rider = False
    for o in all_objects:
        if o.get('cls', '').lower() != 'person':
            continue
        if o.get('conf', 0) < 0.20:
            continue
        px1, py1, px2, py2 = o['xyxy']
        ph = py2 - py1
        pcx = (px1 + px2) / 2
        x_overlap = max(0, min(x2, px2) - max(x1, px1))

        # 4가지 조건 모두 만족해야 라이더:
        #   (a) 가로 중심이 바이크 박스 안
        #   (b) 발 위치(py2)가 바이크 위쪽 55% 안 (안장 부근)
        #   (c) 사람 키 > 바이크 높이의 45% (작은 보행자 제외)
        #   (d) 박스 가로 겹침 > 30%
        if (x1 <= pcx <= x2
                and py2 < y1 + mh_box * 0.55
                and ph > mh_box * 0.45
                and x_overlap > mw_box * 0.3):
            has_rider = True
            break
    if has_rider:
        return False  # 라이더 있음 → 주행 중

    # ──────────────────────────────────────────────
    # [2] DA (도로) 비율 — v8b.3 NEW
    # ──────────────────────────────────────────────
    # 박스 하단 50% (바퀴 위치) 영역의 도로 비율
    # 도로 위에 충분히 있지 않으면 주차 가능성 높음
    mh_d = da_masks_e.shape[1]
    mw_d = da_masks_e.shape[2]
    dx1 = max(0, int(x1 * mw_d / orig_w))
    dx2 = min(mw_d, int(x2 * mw_d / orig_w))
    dy_top = y1 + mh_box * 0.50
    dy1 = max(0, int(dy_top * mh_d / orig_h))
    dy2 = min(mh_d, int(y2 * mh_d / orig_h))

    road_pct = get_pct_smooth(da_masks_e, min(fi, (yn_lim or len(da_masks_e)) - 1),
                              dx1, dy1, dx2, dy2, window=2)
    if road_pct < road_thr:
        return True  # 비도로 = 주차

    # ──────────────────────────────────────────────
    # [3] Sidewalk 비율 (B5 dilate 마스크)
    # ──────────────────────────────────────────────
    mh_s = side_masks_d.shape[1]
    mw_s = side_masks_d.shape[2]
    sx1 = max(0, int(x1 * mw_s / orig_w))
    sx2 = min(mw_s, int(x2 * mw_s / orig_w))
    sy1 = max(0, int(y1 * mh_s / orig_h))
    sy2 = min(mh_s, int(y2 * mh_s / orig_h))

    side_pct = get_pct_smooth(side_masks_d, min(fi, (sn_lim or len(side_masks_d)) - 1),
                              sx1, sy1, sx2, sy2, window=2)
    if side_pct > side_thr:
        return True  # 인도 위 = 주차

    # ──────────────────────────────────────────────
    # [4] 화면 가장자리 fallback
    # ──────────────────────────────────────────────
    # sidewalk/DA 둘 다 못 잡았어도 가장자리면 주차로 추정
    cx_ratio = (x1 + x2) / 2 / orig_w
    if cx_ratio < edge_zone or cx_ratio > (1 - edge_zone):
        return True

    return False  # 다 통과 → 주행 중

print('✅ is_parked_v4 정의 완료')

## 5️⃣ HUD 헬퍼 (폰트 / 색상 / 카드)

In [ ]:
# ============================================================
# HUD 헬퍼
# ============================================================
def _font(size):
    """시스템 폰트 로드 (DejaVu Bold 우선)"""
    for p in ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

F_XS, F_S, F_M = _font(18), _font(22), _font(28)

# HUD 색상 팔레트 (Midnight Tech)
RGB = {
    'cyan':   (0, 220, 255),
    'orange': (255, 140, 30),
    'red':    (255, 50, 50),
    'white':  (255, 255, 255),
    'gray':   (180, 180, 180),
    'pink':   (255, 100, 150),
    'blue':   (80, 140, 255),
    'bgdark': (10, 20, 40),
}

def draw_card(d, x, y, w_, h_, header, hcol, lines, alpha=215):
    """HUD 카드 그리기 (제목 + 본문 라인들)"""
    d.rectangle([x, y, x + w_, y + h_],
                fill=RGB['bgdark'] + (alpha,),
                outline=hcol + (255,), width=2)
    d.text((x + 12, y + 8), header, font=F_S, fill=hcol + (255,))
    for i, line in enumerate(lines):
        col = RGB['white'] if i == 0 else RGB['gray']
        d.text((x + 12, y + 38 + i * 28), line, font=F_S, fill=col + (255,))

print('✅ HUD 헬퍼 준비 완료')

## 6️⃣ 메인 렌더 — 클래스별 차등 적용

| 클래스 | 처리 |
|---|---|
| `motorcycle`, `bicycle` | `is_parked_v4` 체크 → 정지면 제외 |
| `person` | **무조건 표시** (인도/차도/정지 무관) |
| `car`, `truck`, `bus` | **무조건 표시** (자동차 안 사람은 차 박스로 커버) |

박스 색상:
- 🟠 motorcycle (주행 중) — 주황
- 🩷 person — 핑크
- 🩵 car — 시안
- 🟦 truck/bus — 파랑

In [ ]:
# ============================================================
# 메인 렌더 루프 (v8b.3)
# ============================================================
COLOR = {
    'motorcycle': RGB['orange'],   # 주행 중 바이크 → 주황
    'car':        RGB['cyan'],     # 차 → 시안
    'truck':      RGB['blue'],
    'bus':        RGB['blue'],
    'person':     RGB['pink'],     # 사람 → 핑크
    'bicycle':    RGB['orange'],
}
LABELS = {'motorcycle': 'BIKE', 'car': 'CAR', 'truck': 'TRUCK', 'bus': 'BUS',
          'person': 'PERSON', 'bicycle': 'BIKE'}

# v8b.3 임계값
SIDE_THR  = 0.10   # sidewalk 비율 임계
EDGE_ZONE = 0.20   # 가장자리 좌/우 20%
ROAD_THR  = 0.40   # 도로 비율 임계 (이하 → 비도로 = 주차)
CONF_THR  = 0.20   # YOLO confidence 임계

cap = cv2.VideoCapture(str(VIDEO))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer = cv2.VideoWriter(str(OUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'),
                         fps, (orig_w, orig_h))

stats = {'kept': {'motorcycle': 0, 'car': 0, 'person': 0,
                  'truck': 0, 'bus': 0, 'bicycle': 0},
         'parked_filtered': 0, 'frames': 0}

t0 = time.time()
fi = 0
print(f"\n🎬 v8b.3 렌더 시작 (총 {total} 프레임)\n" + "="*70)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).convert('RGBA')
    overlay = Image.new('RGBA', (orig_w, orig_h), (0, 0, 0, 0))
    d = ImageDraw.Draw(overlay)

    yolo_frame = yolo['frames'][min(fi, n_frames_yolo - 1)]
    objects = yolo_frame.get('objects', [])

    # ===== 객체 처리 (클래스별 차등) =====
    cur_counts = {k: 0 for k in LABELS}
    for o in objects:
        cls = o.get('cls', '').lower()
        if cls not in COLOR:
            continue
        if o.get('conf', 0) < CONF_THR:
            continue

        box = o['xyxy']
        x1, y1, x2, y2 = [int(v) for v in box]
        if x2 - x1 < 12 or y2 - y1 < 12:
            continue

        # ★ 클래스별 차등 ★
        if cls in ('motorcycle', 'bicycle'):
            # 바이크만 주차 판정
            if is_parked_v4(box, objects, side_masks_dilated, da_eroded, fi,
                            orig_w, orig_h,
                            side_thr=SIDE_THR, edge_zone=EDGE_ZONE, road_thr=ROAD_THR,
                            sn_lim=sn, yn_lim=yn):
                stats['parked_filtered'] += 1
                continue   # 정지 바이크 → 표시 X
        # person/car/truck/bus는 검사 없이 무조건 표시

        # 박스 그리기
        col = COLOR[cls]
        d.rectangle([x1, y1, x2, y2], outline=col + (255,), width=3)
        d.text((x1, y1 - 22), LABELS[cls], font=F_XS, fill=col + (255,))
        cur_counts[cls] += 1
        stats['kept'][cls] = stats['kept'].get(cls, 0) + 1

    # ===== HUD =====
    # 헤더 바
    d.rectangle([0, 0, orig_w, 50], fill=RGB['bgdark'] + (220,))
    d.text((20, 14), 'GUARDIAN | DTM', font=F_M, fill=RGB['cyan'] + (255,))
    rec = f"REC {int(fi/fps):02d}s"
    rb = d.textbbox((0, 0), rec, font=F_M)[2]
    d.text((orig_w - rb - 20, 14), rec, font=F_M, fill=RGB['red'] + (255,))

    # 시나리오 라벨 (상단 중앙)
    sub = "Vietnam Night Market - DA + Sidewalk Filter v4"
    sw2 = d.textbbox((0, 0), sub, font=F_S)[2]
    d.rectangle([(orig_w - sw2)//2 - 10, 60, (orig_w + sw2)//2 + 10, 90], fill=(0, 0, 0, 200))
    d.text(((orig_w - sw2)//2, 64), sub, font=F_S, fill=RGB['cyan'] + (255,))

    # ENVIRONMENT 카드 (좌상단)
    draw_card(d, 25, 110, 320, 150, 'ENVIRONMENT', RGB['red'],
              ['DENSE TRAFFIC', 'PARKED BIKES IGNORED', 'PEDESTRIANS DETECTED'])

    # DETECTED OBJECTS 카드 (우상단)
    det_lines = []
    for cls, cnt in cur_counts.items():
        if cnt == 0:
            continue
        det_lines.append(f"[{LABELS[cls]:<6}] x{cnt}")
    if not det_lines:
        det_lines = ['(none)']
    card_h = 60 + 28 * len(det_lines)
    draw_card(d, orig_w - 355, 110, 330, card_h, 'DETECTED OBJECTS', RGB['cyan'], det_lines)

    # 푸터: 필터 통계
    foot = f"Parked filtered: {stats['parked_filtered']}"
    d.rectangle([0, orig_h - 40, orig_w, orig_h], fill=RGB['bgdark'] + (180,))
    d.text((20, orig_h - 32), foot, font=F_S, fill=RGB['gray'] + (255,))

    # 합성 + 출력
    out_img = Image.alpha_composite(img, overlay).convert('RGB')
    out_frame = cv2.cvtColor(np.array(out_img), cv2.COLOR_RGB2BGR)
    writer.write(out_frame)

    fi += 1
    stats['frames'] = fi
    if fi % 100 == 0:
        elapsed = time.time() - t0
        eta = elapsed * (total - fi) / max(fi, 1)
        print(f"  [{fi:4d}/{total}] {elapsed:.0f}s, ETA {eta:.0f}s | filtered: {stats['parked_filtered']}")

cap.release()
writer.release()

# ===== 최종 통계 =====
print("="*70)
print(f"🎉 완료 — {(time.time()-t0)/60:.1f}분")
print(f"📁 출력: {OUT_PATH} ({OUT_PATH.stat().st_size/1e6:.1f}MB)")
print(f"\n📊 통계:")
print(f"   처리 프레임             : {stats['frames']}")
print(f"   필터된 정지 바이크        : {stats['parked_filtered']}")
print(f"   유지된 객체:")
for cls, cnt in stats['kept'].items():
    if cnt > 0:
        print(f"     {cls:<12} {cnt:>6}")
print(f"\n📌 진화 비교:")
print(f"   v8b   (motion만)             : 1,159건  (4%)")
print(f"   v8b.2 (+ sidewalk + 위치)    : 14,769건 (50%)")
print(f"   v8b.3 (+ DA + temporal)      : {stats['parked_filtered']:,}건  (66%)")

## 7️⃣ 결과 미리보기 (옵션)

In [ ]:
# ============================================================
# 결과 영상 미리보기
# ============================================================
from IPython.display import Video, display
display(Video(str(OUT_PATH), embed=True, width=900))

## 📝 결론 및 한계

### 사용한 모델 4개
1. **YOLO11m-seg** (Ultralytics) — 객체 탐지
2. **SegFormer-B5** (`nvidia/segformer-b5-finetuned-cityscapes-1024-1024`) — 인도 분할
3. **YOLOPv2** (`CAIC-AD/YOLOPv2`) — 도로/차선 분할
4. **Farneback Optical Flow** (OpenCV) — 모션 (v8b만 사용, 후속 폐기)

### 알고리즘 핵심 — 클래스별 차등
- **motorcycle/bicycle**: `is_parked_v4` 점수제 → 정지면 제외
- **person/car/truck/bus**: 무조건 표시

### v8b.3 알고리즘 (motion-free)
1. 라이더 4중 체크 → 라이더 있으면 주행
2. **DA(YOLOPv2) 도로 비율 < 40% → 주차** (NEW)
3. Sidewalk(SegFormer-B5 dilate) > 10% → 주차
4. 화면 가장자리 좌/우 20% → 주차 (fallback)
5. **±2 프레임 temporal smoothing** (NEW)

### 결과
- 총 motorcycle 객체 29,327건 중 **19,487건 (66%) 정지로 판정 → 필터**
- person 26,407건 그대로 보존 (보행자 안 잘림)
- car/truck/bus도 보존
- 렌더 시간: 5.6분 / 28초 영상 (Colab Pro A100)

### 한계 및 향후 과제
- **100% 정지 바이크 잡기 불가능**: 야간 + 군중 + 노점 가림으로 SegFormer/YOLOPv2 모두 한계
- **SAM (Segment Anything)** 사용하면 95%+ 가능하지만 1시간+ 추가 추론 필요 → 시제품 데모 비용 대비 효율 낮음
- 발표/시제품용으로는 **v8b.3 (66%)**이 적정선
- 향후 모바일 폰 실시간 버전 (`guardian_v6_5.html`)은 더 가벼운 YOLOv8n + OpenCV.js 조합으로 재구현됨